# Lenormand B4-E2 — Candidate Meta-Calibrator + Adaptive Event Set

B4-E2 不加载、不训练 27B。它复用 Fold-0 的 57,977 个候选及 margin，训练一个很小的候选相关性校准器：

```text
27B frozen evidence candidates
        ↓
candidate observable features (margin/source/length/rank/risk)
        ↓
cross-fitted logistic relevance calibrator
        ↓ probability ≥ 0.40
literal dedup + 40-char event suppression + risk safety caps
        ↓
variable-cardinality evidence set
```

新的对照不是旧 top-3，而是更强的 `Indicator → []` baseline。预注册 Gate：

- strict user-grouped cross-fit Evidence F1 ≥ 0.747；
- 相对 Indicator-empty strong baseline ≥ +0.015；
- 至少 2/3 inner folds 获胜。

预计 CPU 1–5 分钟；不需要 GPU。通过后冻结所有参数，只去 outer Fold 1/2 确认。

In [ ]:
#@title 0. Python 3.13 科学计算栈修复（若自动重启，重连后从本格再运行）
import importlib.metadata as metadata
import os, subprocess, sys

PINNED = {
    'numpy': '2.3.4',
    'scipy': '1.16.3',
    'pandas': '2.3.3',
    'scikit-learn': '1.7.2',
}
installed = {}
for package in PINNED:
    try:
        installed[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed[package] = None

if installed != PINNED:
    print('Repairing binary stack:', installed, '->', PINNED)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
        '--force-reinstall', '--only-binary=:all:',
        'numpy==2.3.4', 'scipy==1.16.3', 'pandas==2.3.3',
        'scikit-learn==1.7.2',
    ])
    print('Install complete. Kernel will restart; reconnect and rerun this cell.')
    os.kill(os.getpid(), 9)
else:
    import numpy as np, pandas as pd, scipy, sklearn
    print({'python': sys.version.split()[0], **installed, 'binary_stack': 'PASS'})

In [ ]:
#@title 1. Drive、模块和 Fold-0 输入
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import importlib, json, shutil, sys
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
FOLD_REFERENCE = ROOT / 'results' / 'B4P_AVC_FAST3' / 'B4P_CORE_OOF.npz'
TASK1_EVAL_DIR = (
    ROOT / 'results' / 'B4_TASK1_Q38_FULL64_FOLD0' /
    'Q38_FULL64' / 'fold_0' / 'EVALUATION'
)
AUDIT_PATH = TASK1_EVAL_DIR / 'evidence_candidate_audit.csv'
PREDICTION_PATH = TASK1_EVAL_DIR / 'validation_predictions.csv'
ARTIFACT_ROOT = ROOT / 'results' / 'B4E2_CANDIDATE_META_FOLD0'

MODULES = {
    'b1_experiments.py': None,
    'b1_innovation_experiments.py': None,
    'b4p_anchor_verifier.py': None,
    'qwen38_dual_task_experiments.py': 'Q38_RUNTIME_REVISION = \"2026-08-24.official-evidence-scorer-v3\"',
    'b4e_evidence_set.py': 'B4E_RUNTIME_REVISION = \"2026-08-24.official-one-to-one-event-set-v1\"',
    'b4e_candidate_meta.py': 'B4E2_RUNTIME_REVISION = \"2026-08-24.candidate-meta-event-decoder-v2\"',
}
ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
stale = []
for name, marker in MODULES.items():
    path = ROOT / name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('请一次性上传并覆盖：', stale)
    uploaded = files.upload()
    for name in stale:
        if name not in uploaded:
            raise FileNotFoundError(name)
        shutil.copy2('/content/' + name, ROOT / name)

if not AUDIT_PATH.exists() or not PREDICTION_PATH.exists():
    print('Drive 未找到 Fold-0 产物，请上传两个 CSV。')
    uploaded = files.upload()
    for name, destination in [
        ('evidence_candidate_audit.csv', AUDIT_PATH),
        ('validation_predictions.csv', PREDICTION_PATH),
    ]:
        if not destination.exists():
            if name not in uploaded:
                raise FileNotFoundError(name)
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2('/content/' + name, destination)

for path in [TRAIN_PATH, FOLD_REFERENCE, AUDIT_PATH, PREDICTION_PATH]:
    assert path.exists(), path
sys.path.insert(0, str(ROOT))
print({'audit': str(AUDIT_PATH), 'predictions': str(PREDICTION_PATH), 'output': str(ARTIFACT_ROOT)})

In [ ]:
#@title 2. 导入和运行时版本锁
import b1_experiments as b1
import qwen38_dual_task_experiments as q38
import b4e_evidence_set as b4e
import b4e_candidate_meta as b4e2
importlib.reload(b1); importlib.reload(q38); importlib.reload(b4e); importlib.reload(b4e2)

assert q38.Q38_RUNTIME_REVISION == '2026-08-24.official-evidence-scorer-v3'
assert b4e.B4E_RUNTIME_REVISION == '2026-08-24.official-one-to-one-event-set-v1'
assert b4e2.B4E2_RUNTIME_REVISION == '2026-08-24.candidate-meta-event-decoder-v2'
print('Runtime revisions: PASS')

In [ ]:
#@title 3. 数据对齐、outer Fold-0 与特征泄漏审计
bundle = b1.load_training_data(ROOT, TRAIN_PATH)
fold_data = np.load(FOLD_REFERENCE, allow_pickle=True)
folds = np.asarray(fold_data['folds']).astype(int)
assert len(folds) == len(bundle.row_ids)

audit_raw = pd.read_csv(AUDIT_PATH)
validation_raw = pd.read_csv(PREDICTION_PATH)
audit, validation = b4e.prepare_artifacts(audit_raw, validation_raw, bundle)

row_to_index = {str(row_id): idx for idx, row_id in enumerate(bundle.row_ids)}
validation_indices = np.array([row_to_index[row_id] for row_id in validation.row_id])
assert np.all(folds[validation_indices] == 0), '输入不是严格 outer Fold 0'
assert set(audit.query_row_idx.astype(int)).issubset(set(validation_indices.tolist()))

# 模型特征只能来自测试时可观察量，禁止 Gold risk / Gold count。
assert 'gold_count' not in b4e2.MODEL_FEATURES
assert 'gold_risk' not in b4e2.MODEL_FEATURES
assert 'candidate_target' not in b4e2.MODEL_FEATURES

print({
    'validation_rows': len(validation),
    'candidate_rows': len(audit),
    'validation_users': validation.user_id.nunique(),
    'features': list(b4e2.MODEL_FEATURES),
    'leakage_audit': 'PASS',
})

In [ ]:
#@title 4. 先建立真正的强 Baseline：Indicator → 空 Evidence
raw_baseline = b4e.baseline_score(validation)
strong_predictions = b4e2.strong_baseline_predictions(validation)
strong_baseline = b4e.score_predictions(validation, strong_predictions)
display(pd.DataFrame([
    {'system': 'OLD_TOP3_DECODER', **raw_baseline},
    {'system': 'INDICATOR_EMPTY_STRONG_BASELINE', **strong_baseline},
]))
assert strong_baseline['f1'] > raw_baseline['f1']
print('Strong-baseline delta:', strong_baseline['f1'] - raw_baseline['f1'])

In [ ]:
#@title 5. 严格 inner user-grouped Candidate Meta cross-fit（CPU）
CFG = b4e2.CandidateMetaConfig(
    logistic_c=0.1,
    probability_threshold=0.40,
    event_gap_chars=40,
    top_k_indicator=0,
    top_k_ideation=2,
    top_k_behavior=3,
    top_k_attempt=2,
    seed=20260824,
)
print('FROZEN BEFORE OUTER CONFIRMATION:', CFG)

decision = b4e2.crossfit_candidate_meta_decoder(
    audit, validation, bundle, ARTIFACT_ROOT,
    config=CFG, n_splits=3, split_seed=20260824,
)
print(json.dumps(decision, ensure_ascii=False, indent=2))

In [ ]:
#@title 6. 分风险、分 Gold 数量诊断
fold_results = pd.read_csv(ARTIFACT_ROOT / 'B4E2_INNER_FOLD_RESULTS.csv')
stratified = pd.read_csv(ARTIFACT_ROOT / 'B4E2_STRATIFIED_METRICS.csv')
display(fold_results)
display(stratified.pivot(index='stratum', columns='system', values='f1'))

print({
    'accepted_for_outer_confirmation': decision['accepted_for_outer_confirmation'],
    'crossfit_f1': decision['candidate_meta_crossfit']['f1'],
    'delta_vs_strong_baseline': decision['delta_crossfit_f1_vs_strong_baseline'],
    'folds_better': decision['inner_folds_better_than_strong_baseline'],
})

In [ ]:
#@title 7. 保守分数换算与冻结纪律
RISK_WF1 = 0.824440344
FACTOR_MACRO_F1 = 0.691954
EVIDENCE_F1 = decision['candidate_meta_crossfit']['f1']
subtask1 = (0.4 * RISK_WF1 + 0.3 * EVIDENCE_F1) / 0.7
composite = 0.4 * RISK_WF1 + 0.3 * EVIDENCE_F1 + 0.3 * FACTOR_MACRO_F1
print({
    'crossfit_evidence_f1': EVIDENCE_F1,
    'estimated_subtask1': subtask1,
    'estimated_composite': composite,
    'gap_to_reference_rank8_0.7615': 0.7615 - composite,
    'next_step': decision['recommended_next_step'],
})

if decision['accepted_for_outer_confirmation']:
    print('PASS：冻结 .40 / gap40 / k0232 / feature list / Logistic C=.1。')
    print('下一步只把 Fold-0 calibrator 应用到 untouched outer Fold 1/2；严禁看结果后改参数。')
else:
    print('FAIL：保留 Indicator-empty baseline，不烧 outer Fold 1/2。')

In [ ]:
#@title 8. 打包轻量结果与冻结模型
archive = shutil.make_archive('/content/B4E2_CANDIDATE_META_FOLD0', 'zip', ARTIFACT_ROOT)
print('Saved:', archive)
print('Frozen model:', ARTIFACT_ROOT / 'B4E2_FOLD0_META_CALIBRATOR.joblib')
print('Frozen config:', ARTIFACT_ROOT / 'B4E2_FROZEN_CONFIG.json')
# files.download(archive)

## 如何解释 B4-E2

它不是另一个 Evidence 生成模型。27B 负责语义候选与 verifier margin；小校准器把 margin、候选来源、长度、帖内排名和三个 risk predictions 编译成“该候选是否构成可计分事件”的概率。固定阈值自然产生 0/1/2/3 个事件，因此它是一个 **adaptive evidence-count observer**。

Fold-0 cross-fit 只决定是否值得进入确认阶段。真正的泛化结论必须来自完全冻结后对 outer Fold 1/2 的一次性评估。